In [ ]:
#install required data
#!pip install hecdss --break-system-packages

In [ ]:
#import required data
import pandas as pd
from hecdss import HecDss
from datetime import datetime

In [ ]:
#open the observed data, select sheet and clip columns
dfRaw = pd.read_excel('../xls/Observed_Head.xlsx', sheet_name='Water Level Head & Temp')
dfRaw = dfRaw.iloc[2:,40:42]
dfRaw.head()

In [ ]:
#import data on a correct format
dfHeads = pd.DataFrame()
dfHeads.index=pd.to_datetime(dfRaw['SW1'])
dfHeads['Head'] = pd.to_numeric(dfRaw['Unnamed: 41']).to_list()
dfHeads.head()

In [ ]:
#plot histogram
dfHeads.plot()

In [ ]:


# 4. Open DSS file and write the Regular/Irregular Time-Series
dss_file = "../Xls/riverStageDataSw1.dss"

dss_pathname = f"/PROJECT_AREA/{b_part}/{c_part}//{e_part}/{f_part}/"

# 4. Open/Create the file and write directly via the C-extension wrapper
with HecDss(dss_file) as dss:
    # Instead of creating an explicit Container object, the modern 
    # tool allows raw entry parameters directly to the write function:
    dss.write_ts(
        pathname=dss_pathname,
        times=times,
        values=values,
        units="METERS",        # Use "FEET" if your HEC-RAS units are US Customary
        data_type="INST-VAL"   # Instantaneous Value
    )

print(f"Successfully created {dss_file} and loaded path {dss_pathname}!")

In [ ]:
from pydsstools.core import TimeSeriesContainer, UNDEFINED
from pydsstools.heclib.dss import HecDss

# 2. Extract arrays
times = dfHeads.index.tolist()
values = dfHeads['Head'].astype(float).tolist()

dss_file = "../Xls/riverStageDataSw1.dss"
pathname = "/THEISNASH/SW1/STAGE//1HOUR/OBSERVED/"

with HecDss.Open(dss_file) as fid:
    count = len(values)
    interval = 1  # positive = regular time series

    tsc = TimeSeriesContainer(pathname, count, interval)
    tsc.start_time = times[0]
    tsc.data_units = "METERS"
    tsc.data_type = "INST"
    tsc.tzid = "UTC"
    tsc.values = values  # UNDEFINED marks missing data

    fid.put_ts(tsc)

In [ ]:
import inspect
from pydsstools.core import TimeSeriesContainer
print(inspect.signature(TimeSeriesContainer.__init__))